## set up virtual environment and library

- numpy,
- scikit learn,
- matplotlib,
- pandas

In [ ]:
%pip install numpy scikit-learn matplotlib pandas wordcloud nltk

In [ ]:
%pip install transformers sentencepiece

In [ ]:
%pip install torch

In [ ]:
%pip install -U ipywidgets jupyter tqdm

In [ ]:
%pip install -U transformers datasets evaluate accelerate

## Project_1

### Task 1: Data Exploration and Insights (15%)
Task Description
Analyze the AG News dataset to understand its structure and characteristics.
Suggested analyses:

- Class distribution
- Average and median text length (overall and per class)
- Most frequent words per class
- Word clouds per class
- Distribution of n-gram frequencies
-  …

Please don’t limit yourself to these examples and try to summarize the data with numbers and
graphs. Please highlight some of the most important findings in your report.

---
Deliverable:

- Summarize key findings and observations in your report
- Highlight patterns that may influence modeling choices

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from wordcloud import WordCloud
from nltk.util import ngrams

In [ ]:
# how data looks like, here is an example
df = pd.read_csv("train.csv")
df.head()

the datasets have 3 attributes, they are Class, Index, Title, Description.
- Class Index ∈ {1,2,3,4}
- Title ∈ String
- Description ∈ String

#### Analyze class distribution

In [ ]:

def analyze_class_distribution(file_name):
    """
        train.csv and test.csv has n piece of data
        each piece of data belongs to a certain class, which is represented as 1, 2, 3, 4

        1 calculate each class has how many piece of data, N_i, i ∈ 1,2,3,4 is noted as result
        2 N_i / Total : percentage of each class
        3 plot the result as bar diagram

    """
    df = pd.read_csv(file_name)
    class_lst = df["Class Index"].to_list()

    abs_accumulated_class = dict()
    for i in class_lst:
        k = abs_accumulated_class.keys()
        if i in k:
            abs_accumulated_class[i] += 1
        else:
            abs_accumulated_class[i] = 1
    relevant_accumulated_class = abs_accumulated_class.copy()

    size_class = len(class_lst)
    for key, values in relevant_accumulated_class.items():
        relevant_accumulated_class[key] = relevant_accumulated_class[key] / size_class

    return relevant_accumulated_class,abs_accumulated_class

def visulize_relevant_class_distribution(X, file_name):
    # plot the result
    plt.figure(figsize=(3, 3))
    bars = plt.bar(
        X.keys(),
        X.values()
    )

    max_val = max(X.values())
    plt.ylim(0, max_val * 1.25)

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2, # put text at middle
            height + max_val * 0.03,     # vertical offset
            f"{height:.2f}",  # content of the text, keep 2 decimal
            ha="center",   # horizonal alignment
            va="bottom",   # vertical alignment
            fontsize=8,
            color="black"
        )

    plt.text(
        0.5, -0.25,
        "Discovery: All classes have equal proportion (25%)",
        ha="center",
        transform=plt.gca().transAxes,
        color = "red"
    )
    plt.xlabel(f"Data of {file_name}")
    plt.ylabel("Proportion")
    plt.title(f"Class Distribution of {file_name}")

    plt.show()

In [ ]:
def run_t1_analyze_class_distribution():
    paths = ["train.csv","test.csv"]
    for p in paths:
        relevant, _ = analyze_class_distribution(p)
        visulize_relevant_class_distribution(relevant, p)

In [ ]:
run_t1_analyze_class_distribution()

#### Analyze Average and median text length (overall and per class)

In [ ]:

def analyse_text_length(file_path):
    """ 
    Text Length is in the column "Description", what you need to do is to 
    1) calculate the length of description of each piece of data,
    2) sum up the length
    3) divide the summed up length with the length of the dataset
    4) calculate the median of the text length, median means
        if len is odd, then median is at the floor(len/2) + 1
        if len is even, then median is at len/2
    5) return the ave length and the median length as the return value

    Parameters:
        file_path: the path of the file which need to be calculated,e.g. train.csv
    Returns:
        ave_len: the average length of each piece of data
        median_len: the median length of each piece of data

    """

    df = pd.read_csv(file_path)
    descriptions = df["Description"].tolist()
    l = len(descriptions)
    description_len = np.array([len(x) for x in descriptions])
    sum_up_description_len = np.sum(description_len)
    median_index = int(np.floor(l/2))
    ave_len = round(sum_up_description_len / l,2)
    median_len = len(descriptions[median_index])
    # print(ave_len)
    # print(median_len)

    return ave_len,median_len


In [ ]:
def run_analyse_text_length():
    paths = ["train.csv","test.csv"]
    for p in paths:
        ave, median = analyse_text_length(p)
        print("file name: ", p, "\n","average text length: ", ave, "\n","median text length: ", median)


print("Here is the discovery of text length analyse")
run_analyse_text_length()


#### Analyze Most frequent words per class

In [ ]:
def analyse_most_frequent_word(file_path):
    """ 
    You have to concatenate the Title and Description, then do analysis.
    how to know which word is most frequent in a class?
    1) collect data of same class into one collection
    2) in each collection of data, count its words frequency, then find the most frequent value
    3) visualize/summarize the result

    I don't know how to calculat them in a smart way
    how to know how many classes there are in the datasets?
    how to collect data by class?

    discovery:
        the most commom word is "the" in each class
    """

    df = pd.read_csv(file_path)
    df["text"] = df["Title"] + " " + df["Description"]
    class_collect = dict()

    ind_max = df.index.max()
    i = 0
    while i <= ind_max :
        dt_class = df["Class Index"][i]
        dt_text = df["text"][i]
        keys = class_collect.keys()
        if(dt_class in keys):
            class_collect[dt_class] = class_collect[dt_class] + " " + dt_text
        else:
            class_collect[dt_class] = dt_text
        i+=1

    frequency_collect = dict()
    for key, value in class_collect.items(): # tuple ("word", frequency)
        c = Counter(value.split()).most_common()[0]
        frequency_collect[key] = c

    for k,v in frequency_collect.items():
        print(k,v)
    


In [ ]:
print("result of train.csv")
analyse_most_frequent_word("train.csv")
print("result of test.csv")
analyse_most_frequent_word("test.csv")
print(f"Discovery: the most commom word in the file train and test is \" the \" in each class")

#### Analyze Word clouds per class

In [ ]:
def get_text_collection_by_class(file_path):
    df = pd.read_csv(file_path)
    df["text"] = df["Title"] + " " + df["Description"]
    class_collect = dict()

    ind_max = df.index.max()
    i = 0
    while i <= ind_max :
        dt_class = df["Class Index"][i]
        dt_text = df["text"][i]
        keys = class_collect.keys()
        if(dt_class in keys):
            class_collect[dt_class] = class_collect[dt_class] + " " + dt_text
        else:
            class_collect[dt_class] = dt_text
        i+=1
    return class_collect

In [ ]:
train_dt_text_collect = get_text_collection_by_class("train.csv")
test_dt_text_collect = get_text_collection_by_class("test.csv")

In [ ]:
def draw_word_could(text_collect,title):
    print(f"word cloud of {title} dataset",title)
    for key, value in text_collect.items():
        wc = WordCloud(background_color="white").generate(value)
        print("class: ", key)
        plt.imshow(wc)
        plt.axis("off")
        plt.show()

draw_word_could(train_dt_text_collect,"train")
print("\n")
draw_word_could(test_dt_text_collect,"test")

#### Analyze Distribution of n-gram frequencies

In [ ]:

""" 
1) generate n-grams
2) calculate n-grams' frequency
3) calculate its frequency's distribution

in the task above, we already have train_dt_text_collect, and test_dt_text_collect,
they have collected text(title+description) by class


what means distribution analyse
    - visualize distribution
    - categorize the distribution
    - statistische Merkmale

"""

def generate_ngram(n:int, token_list:list):
    """ 
    parameters:
        n: n gram's n
        tocken_list: a list of tocken. On this list we should apply ngram
    example:
        input: 
            n: 2
            tocken_list: ["Venezuelans", "Vote", "Early","in", "Referendum"]
        output: [('Venezuelans', 'Vote'), ('Vote', 'Early'), ('Early', 'in'), ('in', 'Referendum')]
    """

    return list(ngrams(token_list,n))
    

def ngram_frequency(n_grams_tocken_lst):
    r = Counter(n_grams_tocken_lst)
    return r

def topk_frequent_ngram(ngram_freq, k):
    return ngram_freq.most_common(k)

def visualize_ngram_distribution(ngram_freq,title):
    freqs = list(ngram_freq.values())
    freqs.sort(reverse=True)
    plt.figure(figsize=(7,2))
    plt.plot(freqs)
    title = "class: " + str(title)
    plt.title(title)
    plt.xlabel("Rank")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
    

def run_ngram_distribution_analyse(dt_text_collect,n):
    for key, value in dt_text_collect.items():
        ngrams = generate_ngram(n,list(value.split()))
        freqs = ngram_frequency(ngrams)
        visualize_ngram_distribution(freqs,key)


In [ ]:
print("Discovery: the ngram frequency follows Long-tail distribution")
run_ngram_distribution_analyse(train_dt_text_collect,4)

### Task 2: Pre-processing (15%)

Task Description

Design and implement a preprocessing pipeline suitable for text modeling.

Possible components include:

- Tokenization  
- Lowercasing  
- Stopword removal  
- Stemming or lemmatization  
- Handling punctuation or numbers  
- Subword tokenization (for Transformer models)

---

Report Requirements

In your report, you should:

- Clearly describe your final preprocessing pipeline  
- Justify which steps you included and which you excluded

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from wordcloud import WordCloud
from nltk.util import ngrams
import nltk
from nltk.tokenize import word_tokenize,RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer,WordNetLemmatizer
import re
from transformers import AutoTokenizer
import torch
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
print("torch version:", torch.__version__)
print("mps available:", torch.backends.mps.is_available())
print("mps built:", torch.backends.mps.is_built())

In [ ]:
"""
After donwload "punkt", and "punkt_tab", you can use word_tokenize 
directly, without "preserve_line=True"

If you not download it, you have to use word_tokenize in this way:
a = "I love NLP, how about You?"
b = word_tokenize(a,preserve_line=True)
"""
nltk.download("punkt")
nltk.download("punkt_tab")  # optional, depends on NLTK version

nltk.download("stopwords")
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# very useful links
# https://www.geeksforgeeks.org/nlp/natural-language-processing-nlp-tutorial/
# https://huggingface.co/docs/transformers/tokenizer_summary

class Preprocessor():

    def __init__(self,file_path: str, transformer_name: str = "bert-base-uncased"):

        """
        the file itself has format
        Class Index, Title, Description
        number        str      str

        parameter:
            file_path: the path of the file, the file is supposed to be a csv file
        """
        self.file_path = file_path
        self.dt_raw = pd.read_csv(file_path)
        self.transformer_name = transformer_name
        self.hf_tokenizer = AutoTokenizer.from_pretrained(transformer_name, use_fast=True) # hf stands for hugging face

    def tockenize(self,text:str) -> list:
        """
        Question:
            nltk.tokenize has many kinds of tokenizer, should we pass tokenizer as parameter?
        """
        return word_tokenize(text)
        
    def lowercase(self,text:str) -> str:
        return text.lower()
    

    def rm_stopword(self, tokens:list[str]):
        """
        This function removes stopwords, suppose the text's language is english.

        what is stop words? 
            Stop words typically fall into these grammatical categories:
            Articles: a, an, the
            Prepositions: in, on, at, of, for, with, about
            Conjunctions: and, but, or, so, because
            Pronouns: I, you, he, she, it, we, they, this, that, these
            Auxiliary Verbs: is, am, are, was, were, be, been, have, has, had, do, does, did
            Common Verbs/Adverbs: can, will, would, should, very, too, just, not (see caution below)
        """
        stop_words = set(stopwords.words("english"))
        filtered_tokens = [
            word for word in tokens
            if word.lower() not in stop_words
            and word.isalpha()   # remove punctuation
        ]
        return filtered_tokens
    
    def stemming(self, tokens:list[str]):
        """
        This function reduces words to their root form, often result in non-valid words.

        Question: 
            what is difference, pros, cons between different stemmers?

        example:
            Original words: ['running', 'jumps', 'happily', 'running', 'happily']
            Stemmed words: ['run', 'jump', 'happili', 'run', 'happili']
        """
        stemmer = PorterStemmer()
        stemmed_tokens = [stemmer.stem(word) for word in tokens]
        return stemmed_tokens
        
    # lemmatization it
    def lemmatize(self,tokens:list[str])  ->list:
        """
        This function reduces words to their base form(lemma), ensuring a valid word.

        example:
            Original Text: The cats were running faster than the dogs.
            Lemmatized Words: ['The', 'cat', 'were', 'running', 'faster', 'than', 'the', 'dog', '.']
        """
        lemmatizer = WordNetLemmatizer()
        lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
        return lemmatized_words
    
    
    def handle_punctuation_and_num(self,text:str)-> list:
        """
        This function handles(removes) punctuation and numbers of the input text.

        Regex:
            \w+           it keeps words + numbers
            [A-Za-z]+     it only keeps letters

        parameters:
            text: a string, we'll handle punctuation and number on this text
        return:
            a list of words, which has excluded punctuation and numbers
        example:
            input text:     "# + , punctuation and numbers like 123."
            output result:  ['punctuation', 'and', 'numbers', 'like']
        """
        tokenizer = RegexpTokenizer(r'[A-Za-z]+')
        tokenized_text_lst = tokenizer.tokenize(text)
        return tokenized_text_lst


    def tokenize_subword_transformer(self,text:str, add_special_tokens: bool = True) -> dict:
        """
        Returns subword tokens and token ids for a Transformer tokenizer.

        add_special_tokens:
          - True: includes [CLS]/[SEP] for BERT-like models
          - False: raw subword pieces only
        """
        ids = self.hf_tokenizer.encode(text, add_special_tokens=add_special_tokens)
        tokens = self.hf_tokenizer.convert_ids_to_tokens(ids)
        return {"tokens": tokens, "ids": ids}
    
    def classic_pipeline(
        self,
        use_title: bool = True,
        use_description: bool = True,
        join_with: str = " [SEP] ",     # 仅作为分隔符，classic里也OK
        use_regexp_tokenizer: bool = True,
        remove_stopwords: bool = True,
        do_stemming: bool = False,
        do_lemmatize: bool = True,
        make_tfidf: bool = True,
        tfidf_ngram_range: tuple = (1, 2),
        tfidf_min_df: int = 2,
        tfidf_max_df: float = 0.95,
    ):
        """
        Classical NLP pipeline:
        - Merge Title + Description into one text per row
        - Lowercase
        - Tokenize (RegexpTokenizer OR NLTK word_tokenize)
        - (optional) remove stopwords + punctuation + numbers
        - (optional) lemmatize OR stem
        - Create clean_text = " ".join(tokens)
        - (optional) TF-IDF vectorization

        Returns:
          df_out, (X_tfidf, vectorizer) if make_tfidf else df_out
        """
        df = self.dt_raw.copy()

        # ---- 1) basic cleaning + merge text per row ----
        for col in ["Title", "Description"]:
            if col in df.columns:
                df[col] = df[col].fillna("").astype(str)

        parts = []
        if use_title and "Title" in df.columns:
            parts.append(df["Title"])
        if use_description and "Description" in df.columns:
            parts.append(df["Description"])

        if not parts:
            raise ValueError("No text columns selected. Check use_title/use_description and CSV headers.")

        if len(parts) == 1:
            df["text"] = parts[0].astype(str).str.strip()
        else:
            df["text"] = parts[0].astype(str).str.strip() + join_with + parts[1].astype(str).str.strip()

        # labels
        if "Class Index" not in df.columns:
            raise ValueError("Column 'Class Index' not found in CSV.")
        df["y"] = df["Class Index"].astype(int)

        # ---- 2) lowercase ----
        df["text"] = df["text"].apply(self.lowercase)

        # ---- 3) tokenize ----
        if use_regexp_tokenizer:
            # regex already removes punctuation/numbers
            df["tokens"] = df["text"].apply(self.handle_punctuation_and_num)
        else:
            # NLTK tokenization keeps punctuation; we'll filter later
            df["tokens"] = df["text"].apply(self.tockenize)

        # ---- 4) optional: remove stopwords + keep alpha ----
        if remove_stopwords:
            df["tokens"] = df["tokens"].apply(self.rm_stopword)
        else:
            # even if not removing stopwords, still remove non-alpha if you used NLTK tokenizer
            if not use_regexp_tokenizer:
                df["tokens"] = df["tokens"].apply(lambda toks: [t for t in toks if t.isalpha()])

        # ---- 5) optional: lemmatize / stem (choose one) ----
        if do_stemming and do_lemmatize:
            raise ValueError("Choose either stemming OR lemmatize, not both.")
        if do_lemmatize:
            df["tokens"] = df["tokens"].apply(self.lemmatize)
        elif do_stemming:
            df["tokens"] = df["tokens"].apply(self.stemming)

        # ---- 6) build clean_text for vectorizer ----
        df["clean_text"] = df["tokens"].apply(lambda toks: " ".join(toks))

        # ---- 7) TF-IDF vectorization (optional) ----
        if not make_tfidf:
            return df[["y", "text", "tokens", "clean_text"]]

        vectorizer = TfidfVectorizer(
            ngram_range=tfidf_ngram_range,
            min_df=tfidf_min_df,
            max_df=tfidf_max_df,
            sublinear_tf=True,
            max_features=20000,
        )
        X_tfidf = vectorizer.fit_transform(df["clean_text"])

        return df[["y", "text", "tokens", "clean_text"]], X_tfidf, vectorizer
    
    def transformer_based_pipeline(
        self,
        use_title: bool = True,
        use_description: bool = True,
        join_with: str = " [SEP] ",
        max_length: int = 128,
        padding: str = "max_length",     # "max_length" 或 True (longest)
        truncation: bool = True,
        return_tensors: str | None = None,  # None / "pt" / "tf" / "np"
    ):
        """
        Transformer-based NLP pipeline:
        - Merge Title + Description into one text per row
        - Minimal cleaning (fillna, strip)
        - Use HuggingFace tokenizer to produce:
            input_ids, attention_mask, (optional token_type_ids)
        - Return labels y and encoded inputs

        Returns:
        df_out, encoded
            df_out: columns ["y", "text"]
            encoded: dict with keys like input_ids, attention_mask, token_type_ids (depending on model)
        """
        df = self.dt_raw.copy()

        # ---- 1) basic cleaning + merge text per row ----
        for col in ["Title", "Description"]:
            if col in df.columns:
                df[col] = df[col].fillna("").astype(str)

        parts = []
        if use_title and "Title" in df.columns:
            parts.append(df["Title"].astype(str).str.strip())
        if use_description and "Description" in df.columns:
            parts.append(df["Description"].astype(str).str.strip())

        if not parts:
            raise ValueError("No text columns selected. Check use_title/use_description and CSV headers.")

        if len(parts) == 1:
            df["text"] = parts[0]
        else:
            df["text"] = parts[0] + join_with + parts[1]

        # ---- 2) labels ----
        if "Class Index" not in df.columns:
            raise ValueError("Column 'Class Index' not found in CSV.")
        df["y"] = df["Class Index"].astype(int)

        # ---- 3) tokenizer encode (batch) ----
        texts = df["text"].tolist()

        encoded = self.hf_tokenizer(
            texts,
            add_special_tokens=True,
            padding=padding,
            truncation=truncation,
            max_length=max_length,
            return_tensors=return_tensors,
        )

        # df_out contains minimal columns; encoded carries model inputs
        df_out = df[["y", "text"]].copy()
        return df_out, encoded
    

#### test & benchmark classcial pipeline

In [ ]:
p = Preprocessor("train.csv")

df_out, X, vec = p.classic_pipeline(
    use_regexp_tokenizer=True,     
    remove_stopwords=True,
    do_lemmatize=True,
    do_stemming=False,
    make_tfidf=True,
    tfidf_ngram_range=(1,2)
)

print(df_out.head())
print(X.shape)
print(vec.get_feature_names_out()[:20])


In [ ]:
df_out, X, vec = p.classic_pipeline()
y = df_out["y"].values

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
print("acc:", clf.score(X_test, y_test))


#### test & benchmark transformer-based pipeline

In [ ]:
p = Preprocessor("train.csv")

df_out, enc = p.transformer_based_pipeline(
    max_length=64,
    return_tensors=None
)

print(df_out.head())
print(enc.keys())                # input_ids, attention_mask, ...
print(len(enc["input_ids"]))     # sample size
print(enc["input_ids"][0][:20])  # the first sample's token ids


### Task 3: Classical Language Modeling (20%)

#### Task 3.1 – N-gram Language Models
For this task you should implement a bigram and a trigram language model
Requirements:

- Compute smoothed probabilities (e.g., Laplace smoothing)
- Generate short text sequences from each model
- Compute and compare perplexity across n-gram sizes

In [ ]:
import math
import re
import random
from collections import Counter
import pandas as pd


class NgramLanguageModel:
    """
    Simple n-gram LM with Laplace (add-alpha) smoothing.

    P(w | context) = (count(context,w) + alpha) / (count(context) + alpha*V)
    """

    def __init__(self, n: int, alpha: float = 1.0, unk_token: str = "<unk>"):
        assert n >= 1
        self.n = n
        self.alpha = alpha
        self.unk = unk_token

        self.vocab = set()
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.V = 0
        self.fitted = False

    @staticmethod
    def tokenize(text: str) -> list[str]:
        """
        A simple tokenizer:
        - lowercase
        - keeps punctuation as separate tokens
        """
        text = (text or "").lower()
        # words with optional apostrophes, numbers, or single punctuation symbols
        return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+|[^\w\s]", text)

    def _prepare_tokens(self, tokens: list[str]) -> list[str]:
        pads = ["<s>"] * (self.n - 1)
        return pads + tokens + ["</s>"]

    def fit(self, texts: list[str]) -> None:
        # 1) build vocabulary from training texts
        for t in texts:
            self.vocab.update(self.tokenize(t))

        # add special tokens
        self.vocab.update({"<s>", "</s>", self.unk})
        self.V = len(self.vocab)

        # 2) count ngrams + contexts
        for t in texts:
            tokens = self.tokenize(t)
            tokens = [tok if tok in self.vocab else self.unk for tok in tokens]
            sent = self._prepare_tokens(tokens)

            for i in range(self.n - 1, len(sent)):
                ngram = tuple(sent[i - self.n + 1 : i + 1])
                ctx = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[ctx] += 1

        self.fitted = True

    def prob(self, word: str, context: list[str]) -> float:
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")

        if word not in self.vocab:
            word = self.unk

        if len(context) != self.n - 1:
            raise ValueError(f"context length must be {self.n - 1} for {self.n}-gram model")

        ctx = tuple(context)
        num = self.ngram_counts[ctx + (word,)] + self.alpha
        den = self.context_counts[ctx] + self.alpha * self.V
        return num / den

    def generate(self, max_tokens: int = 30, seed: int | None = None) -> str:
        """
        Generate a short sequence by sampling from P(next | context).
        """
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")
        if seed is not None:
            random.seed(seed)

        context = ["<s>"] * (self.n - 1)
        out = []

        vocab_list = list(self.vocab)

        for _ in range(max_tokens):
            ctx = tuple(context)
            den = self.context_counts[ctx] + self.alpha * self.V

            weights = [
                (self.ngram_counts[ctx + (w,)] + self.alpha) / den
                for w in vocab_list
            ]
            w = random.choices(vocab_list, weights=weights, k=1)[0]

            if w == "</s>":
                break

            out.append(w)

            if self.n > 1:
                context = (context + [w])[-(self.n - 1):]

        return " ".join(out)

    def perplexity(self, texts: list[str]) -> float:
        """
        Perplexity = exp( - (1/N) * sum log P(w_i | context_i) )
        """
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")

        logp_sum = 0.0
        N = 0

        for t in texts:
            tokens = self.tokenize(t)
            tokens = [tok if tok in self.vocab else self.unk for tok in tokens]
            sent = self._prepare_tokens(tokens)

            for i in range(self.n - 1, len(sent)):
                ctx = sent[i - self.n + 1 : i]
                w = sent[i]
                p = self.prob(w, ctx)
                logp_sum += math.log(p)
                N += 1

        return math.exp(-logp_sum / N) if N > 0 else float("inf")


In [ ]:
class Task3NgramLanguageModel:
    """
    Wrapper for Task 3.1:
    - load train/test csv
    - train bigram + trigram with Laplace smoothing
    - generate samples
    - compute perplexities
    """

    def __init__(self, alpha: float = 1.0, use_columns=("Title", "Description")):
        self.alpha = alpha
        self.use_columns = use_columns
        self.bigram = NgramLanguageModel(n=2, alpha=alpha)
        self.trigram = NgramLanguageModel(n=3, alpha=alpha)

    def _load_texts(self, csv_path: str) -> list[str]:
        df = pd.read_csv(csv_path)
        for c in self.use_columns:
            if c not in df.columns:
                raise ValueError(f"Missing column '{c}' in {csv_path}. Found: {list(df.columns)}")
        text = df[self.use_columns[0]].fillna("").astype(str)
        for c in self.use_columns[1:]:
            text = text + " " + df[c].fillna("").astype(str)
        return text.tolist()

    def train(self, train_csv: str) -> None:
        train_texts = self._load_texts(train_csv)
        self.bigram.fit(train_texts)
        self.trigram.fit(train_texts)

    def evaluate(self, test_csv: str) -> dict:
        test_texts = self._load_texts(test_csv)
        pp2 = self.bigram.perplexity(test_texts)
        pp3 = self.trigram.perplexity(test_texts)
        return {"bigram_perplexity": pp2, "trigram_perplexity": pp3}

    def demo_generation(self, k: int = 3, max_tokens: int = 30, seed: int = 0) -> dict:
        bigram_samples = [self.bigram.generate(max_tokens=max_tokens, seed=seed + i) for i in range(k)]
        trigram_samples = [self.trigram.generate(max_tokens=max_tokens, seed=seed + i) for i in range(k)]
        return {"bigram": bigram_samples, "trigram": trigram_samples}

In [ ]:
task = Task3NgramLanguageModel(alpha=1.0)  # Laplace smoothing (alpha=1)
task.train("train.csv")

print(task.demo_generation(k=3, max_tokens=25, seed=42))

results = task.evaluate("test.csv")
print(results)

#### Task 3.2 – Comparison with a Neural Language Model
Build a simple neural language model consisting of:

- An embedding layer
- One LSTM or GRU layer
- A softmax output layer

Then compare against n-gram models on:

- Perplexity
- Quality of generated text
- And, Training time

### Task 4: Transformer Fine-tuning (30%)

Fine-tune a lightweight transformer model on the dataset.
Suggested model:

- DistilBERT

Requirements:

- Split the data into train/validation/test split
- Implement at least two tokenization approaches (e.g., WordPiece vs. Byte-Pair Encoding)
- Evaluation metrics:

- Accuracy
- Macro-F1
- Confusion matrix

In [ ]:
# https://www.acm.org/publications/proceedings-template

### Task 5: Model Evaluation & Benchmarking (20%)
Evaluate your models using multiple perspectives. First, randomly subsample 100 test instances and
evaluate your model on them, and test different evaluation approaches:

- Metric-based evaluation (accuracy, F1, perplexity) - like a standard evaluation as above, using
the gold labels
- Human evaluation (you & teammates give ratings) - this would represent a scenario in which
you would not have any gold label, and you need to create the ground truth by yourself
- LLM-as-a-judge evaluation (use any local small LLM or API-free model like Llama 3.1 8B local)
    - this would represent a scenario in which, again, no gold labels would be available, but
instead of annotating them manually, you use an LLM to judge the results.

Analyze differences between gold labels across the three different evaluations.

### Task 6 Report Requirements (3–4 pages)
Your report must include:

- Clear explanation of pipeline design
- Key findings in each task
- All evaluation results and metrics
- Hyperparameters for all models
- Challenges and limitations